In [6]:
import oracledb
import pandas as pd
import numpy as np
import time
from datetime import datetime, timedelta
from sqlalchemy import create_engine, text
import importlib

import funciones_carga as funciones_carga
import funciones_recocido as funciones_recocido
#import restricciones as restricciones
#importlib.reload(funciones_carga)
#importlib.reload(funciones_recocido)
importlib.reload(funciones)
importlib.reload(funciones_carga)

<module 'funciones_carga' from 'C:\\Users\\Administrador\\TFM\\funciones_carga.py'>

In [7]:
try:
    matriz_distancias = funciones_carga.cargar_datos_distancias()
    tareas_recogida = funciones_carga.cargar_alumnos()    
    #funciones_carga.visualizar_diccionario_generico(matriz_distancias, n=5)    
    #funciones_carga.visualizar_diccionario_generico(tareas_recogida, n=5)    
    print("\n🔥 ¡Datos listos!")
except Exception as e:
    print(f"❌ Error al cargar datos: {e}")

✅ Matriz cargada: 4761 registros
✅ Cargadas 95 tareas de recogida independientes.

🔥 ¡Datos listos!


In [8]:
# Hallar Solución Inicial
tareas_recogida2 = pd.DataFrame(tareas_recogida)
tareas_recogida_div = (funciones_recocido.dividir_tareas_exceso_capacidad(tareas_recogida2))

# SOLUCIÓN INICIAL
solucion_inicial = funciones_recocido.construir_solucion_inicial(tareas_recogida_div,matriz_distancias)

# MOSTRAR
funciones_recocido.mostrar_solucion_inicial(solucion_inicial,tareas_recogida_div, matriz_distancias)


RESUMEN SOLUCIÓN INICIAL
    RUTA      CENTRO  NUM_PARADAS  ALUMNOS  BUS  OCUPACION_%  TIEMPO_MIN  \
0      1  C-18006728            1        1   38         2.63           5   
1      2  C-18007022            2       49   55        89.09          25   
2      3  C-18007022            6       55   55       100.00          59   
3      4  C-18007022            5       25   38        65.79          65   
4      5  C-18007034            6       40   55        72.73          24   
5      6  C-18007046            5       18   38        47.37          84   
6      7  C-18007058            4       51   55        92.73          38   
7      8  C-18007083            4       44   55        80.00          89   
8      9  C-18007083            5       11   38        28.95          68   
9     10  C-18009353            3       19   38        50.00          56   
10    11  C-18009353            3       11   38        28.95          52   
11    12  C-18009353            7       14   38        36.84  

,RUTA,CENTRO,NUM_PARADAS,ALUMNOS,BUS,OCUPACION_%,TIEMPO_MIN,DISTANCIA_KM,COSTE_€
0,1,C-18006728,1,1,38,2.63,5,2.90,96.14
1,2,C-18007022,2,49,55,89.09,25,12.40,119.86
2,3,C-18007022,6,55,55,100.00,59,44.93,119.86
3,4,C-18007022,5,25,38,65.79,65,48.66,124.98
4,5,C-18007034,6,40,55,72.73,24,14.33,119.86
5,6,C-18007046,5,18,38,47.37,84,76.68,124.98
6,7,C-18007058,4,51,55,92.73,38,22.70,119.86
7,8,C-18007083,4,44,55,80.00,89,96.03,155.82
8,9,C-18007083,5,11,38,28.95,68,53.40,124.98
9,10,C-18009353,3,19,38,50.00,56,50.48,96.14


In [9]:
# ==========================================================
# EXPORTAR SOLUCIÓN INICIAL A EXCEL MULTIHOJA
# ==========================================================

import pandas as pd

# ==========================================================
# LISTA DETALLE
# ==========================================================

datos_detalle = []

# ==========================================================
# LISTA RESUMEN
# ==========================================================

datos_resumen = []

# ==========================================================
# RECORRER RUTAS
# ==========================================================

for ruta in solucion_inicial:

    # ------------------------------------------------------
    # DATOS GENERALES RUTA
    # ------------------------------------------------------

    ruta_id = ruta["id"]

    centro = ruta["centro"]

    alumnos_ruta = ruta["alumnos"]

    bus = ruta["bus"]

    ocupacion = ruta["ocupacion"]

    tiempo = ruta["tiempo"]

    distancia = ruta["distancia"]

    coste = ruta["coste"]

    num_paradas = len(
        ruta["paradas"]
    )

    # ======================================================
    # RESUMEN
    # ======================================================

    datos_resumen.append({

        "RUTA": ruta_id,

        "CENTRO": centro,

        "NUM_PARADAS": num_paradas,

        "ALUMNOS_RUTA": alumnos_ruta,

        "BUS": bus,

        "OCUPACION_%": ocupacion,

        "TIEMPO_MIN": tiempo,

        "DISTANCIA_KM": distancia,

        "COSTE_€": coste
    })

    # ======================================================
    # DETALLE PARADAS
    # ======================================================

    for orden, parada in enumerate(

        ruta["paradas"],

        start=1
    ):

        datos_detalle.append({

            "RUTA": ruta_id,

            "CENTRO": centro,

            "ORDEN": orden,

            "PARADA": parada["parada"],

            "ALUMNOS_PARADA": parada["alumnos"],

            "ALUMNOS_RUTA": alumnos_ruta,

            "BUS": bus,

            "OCUPACION_%": ocupacion,

            "TIEMPO_MIN": tiempo,

            "DISTANCIA_KM": distancia,

            "COSTE_€": coste
        })

# ==========================================================
# DATAFRAMES
# ==========================================================

df_resumen = pd.DataFrame(
    datos_resumen
)

df_detalle = pd.DataFrame(
    datos_detalle
)

# ==========================================================
# EXPORTAR EXCEL MULTIHOJA
# ==========================================================

nombre_fichero = "solucion_inicial.xlsx"

with pd.ExcelWriter(

    nombre_fichero,

    engine="openpyxl"

) as writer:

    # ------------------------------------------------------
    # HOJA RESUMEN
    # ------------------------------------------------------

    df_resumen.to_excel(

        writer,

        sheet_name="Resumen_Rutas",

        index=False
    )

    # ------------------------------------------------------
    # HOJA DETALLE
    # ------------------------------------------------------

    df_detalle.to_excel(

        writer,

        sheet_name="Detalle_Rutas",

        index=False
    )

# ==========================================================
# MENSAJE
# ==========================================================

print(
    f"Excel exportado correctamente: {nombre_fichero}"
)

Excel exportado correctamente: solucion_inicial.xlsx


In [10]:
import itertools
import time
import pandas as pd

# ============================================================
# GENERAR COMBINACIONES DE PARÁMETROS
# ============================================================

def generar_configuraciones_sa():
    """
    Genera configuraciones típicas
    para experimentar con recocido simulado.

    Retorna
    -------
    DataFrame
    """

    # ========================================================
    # ESPACIO PARÁMETROS
    # ========================================================

    temperaturas = [100]

    factores_enfriamiento = [0.90,0.95,0.97]
    factores_enfriamiento = [0.95]

    iteraciones = [75, 90, 100, 110, 125]

    # Parametriza coste
    iteraciones = [75]
    pesos_coste = [1]
    pesos_rutas = [25, 30]
    pesos_ocupacion = [10]        
    pesos_distancia = [0.25, 0.1]

    # Parametriza distancia
    iteraciones = [75]
    pesos_coste = [1]
    pesos_rutas = [5,15]
    pesos_ocupacion = [5]        
    pesos_distancia = [5, 2]

    # ========================================================
    # PRODUCTO CARTESIANO
    # ========================================================

    combinaciones = list(

        itertools.product(

            temperaturas,

            factores_enfriamiento,

            iteraciones,

            pesos_coste,

            pesos_rutas,

            pesos_ocupacion,

            pesos_distancia
        )
    )

    # ========================================================
    # DATAFRAME
    # ========================================================

    df_config = pd.DataFrame(

        combinaciones,

        columns=[

            "temperatura_inicial",

            "factor_enfriamiento",

            "iteraciones_por_temperatura",

            "peso_coste",

            "peso_rutas",

            "peso_ocupacion",

            "peso_distancia"
        ]
    )

    # ========================================================
    # TEMPERATURA MÍNIMA
    # ========================================================
    
    df_config["temperatura_minima"] = 1
    

    return df_config

In [11]:
# APLICACIÓN RECOCIDO

df_config = generar_configuraciones_sa()
#df_config_def = df_config.iloc[[0]]
df_config_def = df_config

# display() renderiza la tabla de forma visual y atractiva
display(df_config)

df_experimentos = funciones_recocido.ejecutar_experimentos_sa(df_config_def, solucion_inicial, tareas_recogida_div, matriz_distancias)

# Ordena por mejor coste
df_experimentos.sort_values(by="mejor_coste").head()

,temperatura_inicial,factor_enfriamiento,iteraciones_por_temperatura,peso_coste,peso_rutas,peso_ocupacion,peso_distancia,temperatura_minima
0,100,0.95,75,1,5,5,5,1
1,100,0.95,75,1,5,5,2,1
2,100,0.95,75,1,15,5,5,1
3,100,0.95,75,1,15,5,2,1



CONFIGURACIÓN 1 / 4
{'temperatura_inicial': 100.0, 'factor_enfriamiento': 0.95, 'iteraciones_por_temperatura': 75.0, 'peso_coste': 1.0, 'peso_rutas': 5.0, 'peso_ocupacion': 5.0, 'peso_distancia': 5.0, 'temperatura_minima': 1.0}
Tiempo de ejecución de configuración: 226.84 s

CONFIGURACIÓN 2 / 4
{'temperatura_inicial': 100.0, 'factor_enfriamiento': 0.95, 'iteraciones_por_temperatura': 75.0, 'peso_coste': 1.0, 'peso_rutas': 5.0, 'peso_ocupacion': 5.0, 'peso_distancia': 2.0, 'temperatura_minima': 1.0}
Tiempo de ejecución de configuración: 288.35 s

CONFIGURACIÓN 3 / 4
{'temperatura_inicial': 100.0, 'factor_enfriamiento': 0.95, 'iteraciones_por_temperatura': 75.0, 'peso_coste': 1.0, 'peso_rutas': 15.0, 'peso_ocupacion': 5.0, 'peso_distancia': 5.0, 'temperatura_minima': 1.0}
Tiempo de ejecución de configuración: 227.56 s

CONFIGURACIÓN 4 / 4
{'temperatura_inicial': 100.0, 'factor_enfriamiento': 0.95, 'iteraciones_por_temperatura': 75.0, 'peso_coste': 1.0, 'peso_rutas': 15.0, 'peso_ocupacio

,temperatura_inicial,factor_enfriamiento,iteraciones_por_temperatura,peso_coste,peso_rutas,peso_ocupacion,peso_distancia,mejor_coste,numero_rutas,ocupacion_media,distancia_total,coste_total,tiempo_total_min,tiempo_ejecucion_s,mejor_solucion,df_resultado,df_historial
1,100.0,0.95,75,1.0,5.0,5.0,2.0,104740.18,26,67.03,831.19,2969.56,1071,288.35,"[{'id': 1, 'centro': 'C-18006728', 'paradas': ...",Metrica Valor 0 ...,ITERACION TEMPERATURA COSTE_ACTUAL ME...
3,100.0,0.95,75,1.0,15.0,5.0,2.0,105015.92,26,68.50,867.67,2967.56,1116,242.21,"[{'id': 1, 'centro': 'C-18006728', 'paradas': ...",Metrica Valor 0 ...,ITERACION TEMPERATURA COSTE_ACTUAL ME...
0,100.0,0.95,75,1.0,5.0,5.0,5.0,107029.70,26,67.16,789.40,2933.60,1057,226.84,"[{'id': 1, 'centro': 'C-18006728', 'paradas': ...",Metrica Valor 0 ...,ITERACION TEMPERATURA COSTE_ACTUAL ME...
2,100.0,0.95,75,1.0,15.0,5.0,5.0,132157.25,27,64.94,782.71,3029.74,1054,227.56,"[{'id': 1, 'centro': 'C-18006728', 'paradas': ...",Metrica Valor 0 ...,ITERACION TEMPERATURA COSTE_ACTUAL ME...


In [14]:
display(df_experimentos.head(20))

# ==========================================================
# EXPORTAR RESULTADOS COMPLETOS A EXCEL MULTIHOJA
# ==========================================================

import pandas as pd

# ==========================================================
# NOMBRE FICHERO
# ==========================================================

nombre_excel = "resultados_experimento_2.xlsx"

# ==========================================================
# CREAR EXCEL
# ==========================================================

with pd.ExcelWriter(
    
    nombre_excel,
    
    engine="openpyxl"
    
) as writer:

    # ======================================================
    # 1. HOJA RESUMEN EXPERIMENTOS
    # ======================================================

    df_resumen = df_experimentos[[
        
        "temperatura_inicial",
        "factor_enfriamiento",
        "iteraciones_por_temperatura",
        
        "peso_coste",
        "peso_rutas",
        "peso_ocupacion",
        "peso_distancia",

        "mejor_coste",
        "numero_rutas",
        "ocupacion_media",
        "distancia_total",
        "coste_total",
        "tiempo_total_min",
        "tiempo_ejecucion_s"

    ]].copy()

    df_resumen = (
        
        df_resumen
        
        .sort_values(
            by="numero_rutas"
        )
        
        .reset_index(drop=True)
    )

    df_resumen.to_excel(
        
        writer,
        
        sheet_name="Resumen",
        
        index=False
    )

    # ======================================================
    # 2. EXPORTAR CADA SOLUCIÓN
    # ======================================================

    for idx, row in df_experimentos.iterrows():

        # --------------------------------------------------
        # NOMBRE HOJA
        # --------------------------------------------------

        nombre_hoja = f"Sol_{idx+1}"

        # --------------------------------------------------
        # DATAFRAME RESULTADO
        # --------------------------------------------------

        df_sol = row["df_resultado"]

        # --------------------------------------------------
        # EXPORTAR
        # --------------------------------------------------

        if isinstance(df_sol, pd.DataFrame):

            df_sol.to_excel(
                
                writer,
                
                sheet_name=nombre_hoja[:31],  # límite Excel
                
                index=False
            )

    # ======================================================
    # 3. EXPORTAR HISTÓRICOS
    # ======================================================

    for idx, row in df_experimentos.iterrows():

        nombre_hoja = f"HIST_{idx+1}"

        df_hist = row["df_historial"]

        if isinstance(df_hist, pd.DataFrame):

            df_hist.to_excel(
                
                writer,
                
                sheet_name=nombre_hoja[:31],
                
                index=False
            )

# ==========================================================
# FIN
# ==========================================================

print(
    f"Excel generado correctamente: {nombre_excel}"
)

,temperatura_inicial,factor_enfriamiento,iteraciones_por_temperatura,peso_coste,peso_rutas,peso_ocupacion,peso_distancia,mejor_coste,numero_rutas,ocupacion_media,distancia_total,coste_total,tiempo_total_min,tiempo_ejecucion_s,mejor_solucion,df_resultado,df_historial
0,100.0,0.95,75,1.0,5.0,5.0,5.0,107029.70,26,67.16,789.40,2933.60,1057,226.84,"[{'id': 1, 'centro': 'C-18006728', 'paradas': ...",Metrica Valor 0 ...,ITERACION TEMPERATURA COSTE_ACTUAL ME...
1,100.0,0.95,75,1.0,5.0,5.0,2.0,104740.18,26,67.03,831.19,2969.56,1071,288.35,"[{'id': 1, 'centro': 'C-18006728', 'paradas': ...",Metrica Valor 0 ...,ITERACION TEMPERATURA COSTE_ACTUAL ME...
2,100.0,0.95,75,1.0,15.0,5.0,5.0,132157.25,27,64.94,782.71,3029.74,1054,227.56,"[{'id': 1, 'centro': 'C-18006728', 'paradas': ...",Metrica Valor 0 ...,ITERACION TEMPERATURA COSTE_ACTUAL ME...
3,100.0,0.95,75,1.0,15.0,5.0,2.0,105015.92,26,68.50,867.67,2967.56,1116,242.21,"[{'id': 1, 'centro': 'C-18006728', 'paradas': ...",Metrica Valor 0 ...,ITERACION TEMPERATURA COSTE_ACTUAL ME...


Excel generado correctamente: resultados_experimento_2.xlsx


In [70]:
# ==========================================================
# EXPORTAR SOLUCIÓN INICIAL A EXCEL MULTIHOJA
# ==========================================================

import pandas as pd

# ==========================================================
# LISTA DETALLE
# ==========================================================

datos_detalle = []

# ==========================================================
# LISTA RESUMEN
# ==========================================================

datos_resumen = []

# ==========================================================
# RECORRER RUTAS
# ==========================================================

for ruta in solucion_inicial:

    # ------------------------------------------------------
    # DATOS GENERALES RUTA
    # ------------------------------------------------------

    ruta_id = ruta["id"]

    centro = ruta["centro"]

    alumnos_ruta = ruta["alumnos"]

    bus = ruta["bus"]

    ocupacion = ruta["ocupacion"]

    tiempo = ruta["tiempo"]

    distancia = ruta["distancia"]

    coste = ruta["coste"]

    num_paradas = len(
        ruta["paradas"]
    )

    # ======================================================
    # RESUMEN
    # ======================================================

    datos_resumen.append({

        "RUTA": ruta_id,

        "CENTRO": centro,

        "NUM_PARADAS": num_paradas,

        "ALUMNOS_RUTA": alumnos_ruta,

        "BUS": bus,

        "OCUPACION_%": ocupacion,

        "TIEMPO_MIN": tiempo,

        "DISTANCIA_KM": distancia,

        "COSTE_€": coste
    })

    # ======================================================
    # DETALLE PARADAS
    # ======================================================

    for orden, parada in enumerate(

        ruta["paradas"],

        start=1
    ):

        datos_detalle.append({

            "RUTA": ruta_id,

            "CENTRO": centro,

            "ORDEN": orden,

            "PARADA": parada["parada"],

            "ALUMNOS_PARADA": parada["alumnos"],

            "ALUMNOS_RUTA": alumnos_ruta,

            "BUS": bus,

            "OCUPACION_%": ocupacion,

            "TIEMPO_MIN": tiempo,

            "DISTANCIA_KM": distancia,

            "COSTE_€": coste
        })

# ==========================================================
# DATAFRAMES
# ==========================================================

df_resumen = pd.DataFrame(
    datos_resumen
)

df_detalle = pd.DataFrame(
    datos_detalle
)

# ==========================================================
# EXPORTAR EXCEL MULTIHOJA
# ==========================================================

nombre_fichero = "solucion_inicial.xlsx"

with pd.ExcelWriter(

    nombre_fichero,

    engine="openpyxl"

) as writer:

    # ------------------------------------------------------
    # HOJA RESUMEN
    # ------------------------------------------------------

    df_resumen.to_excel(

        writer,

        sheet_name="Resumen_Rutas",

        index=False
    )

    # ------------------------------------------------------
    # HOJA DETALLE
    # ------------------------------------------------------

    df_detalle.to_excel(

        writer,

        sheet_name="Detalle_Rutas",

        index=False
    )

# ==========================================================
# MENSAJE
# ==========================================================

print(
    f"Excel exportado correctamente: {nombre_fichero}"
)

Excel exportado correctamente: solucion_inicial.xlsx


In [16]:
import importlib
importlib.reload(funciones_recocido)

print("--- EJECUTANDO RECOCIDO SIMULADO ---")
mejor_sol_sa_j, df_res_sa_j, df_hist_sa_j = funciones_recocido.recocido_simulado(
    solucion_inicial=solucion_inicial,
    tareas_recogida=tareas_recogida_div,  # <--- CAMBIA AQUÍ: Usa el DataFrame original (ej. df_tareas_recogida)
    matriz_distancias=matriz_distancias,
    temperatura_inicial=100,
    temperatura_minima=1,
    factor_enfriamiento=0.95,
    iteraciones_por_temperatura=125,
    peso_coste=0.5,
    peso_rutas=10,
    peso_ocupacion=0.5,
    peso_distancia=0.5,
    mostrar_logs=True
)

# Unificar o limpiar los valores NaN en la tabla comparativa
df_comparativo.loc[df_comparativo["Metrica"] == "Total alumnos", "Algoritmo Genético"] = 836.00

# O de forma general, cambiar los NaN restantes por un guion para que quede elegante
df_comparativo = df_comparativo.fillna("-")
df_experimentos.sort_values(by="mejor_coste").iloc[0]

# ============================================================
# CONSTRUCCIÓN DEL CUADRO COMPARATIVO UNIENDO DATASET
# ============================================================
df_res_sa.columns = ["Metrica", "Recocido Simulado"]
df_res_ga.columns = ["Metrica", "Algoritmo Genético"]

df_comparativo = pd.merge(df_res_sa, df_res_ga, on="Metrica", how="outer")
print("\n========================================================")
print("               TABLA COMPARATIVA DE MÉTODOS")
print("========================================================")
print(df_comparativo)





--- EJECUTANDO RECOCIDO SIMULADO ---
T=95.00 | Mejor coste=18097.83
T=90.25 | Mejor coste=17504.95
T=85.74 | Mejor coste=17504.95
T=81.45 | Mejor coste=17504.95
T=77.38 | Mejor coste=14936.04
T=73.51 | Mejor coste=14923.98
T=69.83 | Mejor coste=14870.25
T=66.34 | Mejor coste=14870.25
T=63.02 | Mejor coste=14870.25
T=59.87 | Mejor coste=14870.25
T=56.88 | Mejor coste=14870.25
T=54.04 | Mejor coste=14870.25
T=51.33 | Mejor coste=14870.25
T=48.77 | Mejor coste=14870.25
T=46.33 | Mejor coste=14870.25
T=44.01 | Mejor coste=14870.25
T=41.81 | Mejor coste=14870.25
T=39.72 | Mejor coste=14870.25
T=37.74 | Mejor coste=14870.25
T=35.85 | Mejor coste=14870.25
T=34.06 | Mejor coste=14870.25
T=32.35 | Mejor coste=14870.25
T=30.74 | Mejor coste=14870.25
T=29.20 | Mejor coste=14869.14
T=27.74 | Mejor coste=14869.14
T=26.35 | Mejor coste=14869.14
T=25.03 | Mejor coste=14869.14
T=23.78 | Mejor coste=14866.20
T=22.59 | Mejor coste=14866.20
T=21.46 | Mejor coste=14866.20
T=20.39 | Mejor coste=14866.20
T=

In [19]:
print("\n--- EJECUTANDO ALGORITMO GENÉTICO ---")
mejor_sol_ga, df_res_ga, df_hist_ga = funciones_recocido.algoritmo_genetico(
    solucion_inicial=solucion_inicial,
    tareas_recogida=tareas_recogida_div,  # <--- CAMBIA AQUÍ: Usa el DataFrame original (ej. df_tareas_recogida)
    matriz_distancias=matriz_distancias,
    tamano_poblacion=30,
    generaciones=50,
    probabilidad_mutacion=0.2,
    peso_coste=1.0,
    peso_rutas=5.0,
    peso_ocupacion=5.0,
    peso_distancia=5.0,
    mostrar_logs=True
)


--- EJECUTANDO ALGORITMO GENÉTICO ---
Gen 1/50 | Mejor Coste: 200111.83 | Coste Medio: 218507.15
Gen 2/50 | Mejor Coste: 200111.83 | Coste Medio: 91962.21
Gen 3/50 | Mejor Coste: 189605.11 | Coste Medio: 42303.59
Gen 4/50 | Mejor Coste: 189605.11 | Coste Medio: 42125.16
Gen 5/50 | Mejor Coste: 189605.11 | Coste Medio: 41641.41
Gen 6/50 | Mejor Coste: 189605.11 | Coste Medio: 55637.27
Gen 7/50 | Mejor Coste: 189605.11 | Coste Medio: 19969.87
Gen 8/50 | Mejor Coste: 189605.11 | Coste Medio: 6320.17
Gen 9/50 | Mejor Coste: 189605.11 | Coste Medio: 6320.17
Gen 10/50 | Mejor Coste: 189605.11 | Coste Medio: 6320.17
Gen 11/50 | Mejor Coste: 189605.11 | Coste Medio: 6320.17
Gen 12/50 | Mejor Coste: 189605.11 | Coste Medio: 6320.17
Gen 13/50 | Mejor Coste: 189605.11 | Coste Medio: 6320.17
Gen 14/50 | Mejor Coste: 189605.11 | Coste Medio: 6320.17
Gen 15/50 | Mejor Coste: 189577.61 | Coste Medio: 12639.42
Gen 16/50 | Mejor Coste: 189577.61 | Coste Medio: 12639.03
Gen 17/50 | Mejor Coste: 189577.

In [86]:
# ============================================================
# EJECUCIÓN COMPARATIVA CORREGIDA: SA vs GA
# ============================================================

import importlib
importlib.reload(funciones_recocido)

print("--- EJECUTANDO RECOCIDO SIMULADO ---")
mejor_sol_sa, df_res_sa, df_hist_sa = funciones_recocido.recocido_simulado(
    solucion_inicial=solucion_inicial,
    tareas_recogida=tareas_recogida_div,  # <--- CAMBIA AQUÍ: Usa el DataFrame original (ej. df_tareas_recogida)
    matriz_distancias=matriz_distancias,
    temperatura_inicial=100,
    temperatura_minima=1,
    factor_enfriamiento=0.95,
    iteraciones_por_temperatura=75,
    peso_coste=1.0,
    peso_rutas=5.0,
    peso_ocupacion=5.0,
    peso_distancia=5.0,
    mostrar_logs=True
)

print("\n--- EJECUTANDO ALGORITMO GENÉTICO ---")
mejor_sol_ga, df_res_ga, df_hist_ga = funciones_recocido.algoritmo_genetico(
    solucion_inicial=solucion_inicial,
    tareas_recogida=tareas_recogida_div,  # <--- CAMBIA AQUÍ: Usa el DataFrame original (ej. df_tareas_recogida)
    matriz_distancias=matriz_distancias,
    tamano_poblacion=30,
    generaciones=50,
    probabilidad_mutacion=0.2,
    peso_coste=1.0,
    peso_rutas=5.0,
    peso_ocupacion=5.0,
    peso_distancia=5.0,
    mostrar_logs=True
)

--- EJECUTANDO RECOCIDO SIMULADO ---
T=95.00 | Mejor coste=184545.04
T=90.25 | Mejor coste=184306.44
T=85.74 | Mejor coste=159030.30
T=81.45 | Mejor coste=158937.78
T=77.38 | Mejor coste=158898.73
T=73.51 | Mejor coste=158706.80
T=69.83 | Mejor coste=158623.48
T=66.34 | Mejor coste=158623.48
T=63.02 | Mejor coste=158623.48
T=59.87 | Mejor coste=133491.75
T=56.88 | Mejor coste=133491.75
T=54.04 | Mejor coste=133491.75
T=51.33 | Mejor coste=133491.75
T=48.77 | Mejor coste=133465.82
T=46.33 | Mejor coste=133465.82
T=44.01 | Mejor coste=133349.60
T=41.81 | Mejor coste=133349.60
T=39.72 | Mejor coste=133349.60
T=37.74 | Mejor coste=133015.76
T=35.85 | Mejor coste=133015.76
T=34.06 | Mejor coste=122611.98
T=32.35 | Mejor coste=114522.73
T=30.74 | Mejor coste=113074.58
T=29.20 | Mejor coste=107739.44
T=27.74 | Mejor coste=107705.07
T=26.35 | Mejor coste=107495.05
T=25.03 | Mejor coste=107495.05
T=23.78 | Mejor coste=107495.05
T=22.59 | Mejor coste=107495.05
T=21.46 | Mejor coste=107495.05
T=2

In [97]:
print("\n--- OTRAS EJECUCIONES ALGORITMO GENÉTICO ---")

mejor_sol_ga_2, df_res_ga_2, df_hist_ga_2 = funciones_recocido.algoritmo_genetico(
    solucion_inicial=solucion_inicial,
    tareas_recogida=tareas_recogida_div,  # <--- CAMBIA AQUÍ: Usa el DataFrame original (ej. df_tareas_recogida)
    matriz_distancias=matriz_distancias,
    tamano_poblacion=30,
    generaciones=20,
    probabilidad_mutacion=0.01,
    peso_coste=1.0,
    peso_rutas=5.0,
    peso_ocupacion=5.0,
    peso_distancia=5.0,
    mostrar_logs=True
)

mejor_sol_ga_3, df_res_ga_3, df_hist_ga_3 = funciones_recocido.algoritmo_genetico(
    solucion_inicial=solucion_inicial,
    tareas_recogida=tareas_recogida_div,  # <--- CAMBIA AQUÍ: Usa el DataFrame original (ej. df_tareas_recogida)
    matriz_distancias=matriz_distancias,
    tamano_poblacion=50,
    generaciones=50,
    probabilidad_mutacion=0.01,
    peso_coste=1.0,
    peso_rutas=5.0,
    peso_ocupacion=5.0,
    peso_distancia=5.0,
    mostrar_logs=True
)

mejor_sol_ga_4, df_res_ga_4, df_hist_ga_4 = funciones_recocido.algoritmo_genetico(
    solucion_inicial=solucion_inicial,
    tareas_recogida=tareas_recogida_div,  # <--- CAMBIA AQUÍ: Usa el DataFrame original (ej. df_tareas_recogida)
    matriz_distancias=matriz_distancias,
    tamano_poblacion=100,
    generaciones=50,
    probabilidad_mutacion=0.01,
    peso_coste=1.0,
    peso_rutas=5.0,
    peso_ocupacion=5.0,
    peso_distancia=5.0,
    mostrar_logs=True
)


mejor_sol_ga_5, df_res_ga_5, df_hist_ga_5 = funciones_recocido.algoritmo_genetico(
    solucion_inicial=solucion_inicial,
    tareas_recogida=tareas_recogida_div,  # <--- CAMBIA AQUÍ: Usa el DataFrame original (ej. df_tareas_recogida)
    matriz_distancias=matriz_distancias,
    tamano_poblacion=200,
    generaciones=10,
    probabilidad_mutacion=0.01,
    peso_coste=6.0,
    peso_rutas=5.0,
    peso_ocupacion=5.0,
    peso_distancia=5.0,
    mostrar_logs=True
)



--- OTRAS EJECUCIONES ALGORITMO GENÉTICO ---
Gen 1/20 | Mejor Coste: 200111.83 | Coste Medio: 218534.62
Gen 2/20 | Mejor Coste: 200111.83 | Coste Medio: 78617.42
Gen 3/20 | Mejor Coste: 200111.83 | Coste Medio: 57984.91
Gen 4/20 | Mejor Coste: 200111.83 | Coste Medio: 21331.40
Gen 5/20 | Mejor Coste: 200111.83 | Coste Medio: 20671.29
Gen 6/20 | Mejor Coste: 200111.83 | Coste Medio: 26681.58
Gen 7/20 | Mejor Coste: 200111.83 | Coste Medio: 26681.58
Gen 8/20 | Mejor Coste: 200111.83 | Coste Medio: 6670.39
Gen 9/20 | Mejor Coste: 200111.83 | Coste Medio: 6670.39
Gen 10/20 | Mejor Coste: 200111.83 | Coste Medio: 13340.79
Gen 11/20 | Mejor Coste: 200111.83 | Coste Medio: 6670.39
Gen 12/20 | Mejor Coste: 200111.83 | Coste Medio: 20011.18
Gen 13/20 | Mejor Coste: 200111.83 | Coste Medio: 26681.58
Gen 14/20 | Mejor Coste: 200111.83 | Coste Medio: 26681.58
Gen 15/20 | Mejor Coste: 200111.83 | Coste Medio: 40022.37
Gen 16/20 | Mejor Coste: 200111.83 | Coste Medio: 6670.39
Gen 17/20 | Mejor Cost

In [104]:
mejor_sol_ga_6, df_res_ga_6, df_hist_ga_6 = funciones_recocido.algoritmo_genetico(
    solucion_inicial=solucion_inicial,
    tareas_recogida=tareas_recogida_div,
    matriz_distancias=matriz_distancias,
    tamano_poblacion=850,             # Más individuos para garantizar diversidad inicial
    generaciones=250,                # Suficientes ciclos para permitir la convergencia
    probabilidad_mutacion=0.20,      # Exploración activa para mover paradas entre rutas
    peso_coste=1.0,
    peso_rutas=20.0,                 # Presión extrema para forzar la reducción de autobuses
    peso_ocupacion=5.0,
    peso_distancia=2.0,              # Permitimos que aumenten temporalmente los km si eso ahorra una ruta
    mostrar_logs=True
)

Gen 1/250 | Mejor Coste: 197097.40 | Coste Medio: 216667.43
Gen 2/250 | Mejor Coste: 191902.06 | Coste Medio: 47458.61
Gen 3/250 | Mejor Coste: 191897.16 | Coste Medio: 26227.08
Gen 4/250 | Mejor Coste: 191897.16 | Coste Medio: 12497.87
Gen 5/250 | Mejor Coste: 191897.16 | Coste Medio: 7149.29
Gen 6/250 | Mejor Coste: 191897.16 | Coste Medio: 4207.43
Gen 7/250 | Mejor Coste: 191897.16 | Coste Medio: 4691.78
Gen 8/250 | Mejor Coste: 191897.16 | Coste Medio: 2712.93
Gen 9/250 | Mejor Coste: 191897.16 | Coste Medio: 2166.74
Gen 10/250 | Mejor Coste: 191897.16 | Coste Medio: 1158.29
Gen 11/250 | Mejor Coste: 191897.16 | Coste Medio: 1187.76
Gen 12/250 | Mejor Coste: 191897.16 | Coste Medio: 225.76
Gen 13/250 | Mejor Coste: 191897.16 | Coste Medio: 451.52
Gen 14/250 | Mejor Coste: 191897.16 | Coste Medio: 451.52
Gen 15/250 | Mejor Coste: 191897.16 | Coste Medio: 451.52
Gen 16/250 | Mejor Coste: 191897.16 | Coste Medio: 677.28
Gen 17/250 | Mejor Coste: 191896.40 | Coste Medio: 1128.81
Gen 18

In [23]:
import pandas as pd
import matplotlib.pyplot as plt

# =====================================================================
# 1. GENERACIÓN DE LA TABLA COMPARATIVA DE PARAMETRIZACIONES
# =====================================================================

def extraer_metricas_ga(df_res, nombre_config):
    """
    Transpone el DataFrame de resultados de cada ejecución para 
    convertir las métricas en filas y las configuraciones en columnas.
    """
    # Copia el DataFrame y asegura que la columna 'Metrica' sea el índice
    df = df_res.copy()
    if 'Metrica' in df.columns:
        df = df.set_index('Metrica')
    
    # Renombrar la columna de valor al nombre de la configuración evaluada
    if 'Valor' in df.columns:
        df = df.rename(columns={'Valor': nombre_config})
    elif 'Algoritmo Genético' in df.columns:
        df = df.rename(columns={'Algoritmo Genético': nombre_config})
    
    return df

# Procesar los 5 DataFrames individuales
df_c1 = extraer_metricas_ga(df_res_ga,   "Configuración 1\n(Pob:30, Gen:50, Pm:0.20)")
"""df_c2 = extraer_metricas_ga(df_res_ga_2, "Configuración 2\n(Pob:30, Gen:20, Pm:0.01)")
df_c3 = extraer_metricas_ga(df_res_ga_3, "Configuración 3\n(Pob:50, Gen:50, Pm:0.01)")
df_c4 = extraer_metricas_ga(df_res_ga_4, "Configuración 4\n(Pob:100, Gen:50, Pm:0.01)")
df_c5 = extraer_metricas_ga(df_res_ga_5, "Configuración 5\n(Pob:200, Gen:10, Pm:0.01)")
df_c6 = extraer_metricas_ga(df_res_ga_6, "Configuración 6\n(Pob:800, Gen:250, Pm:0.01)")
"""

# Consolidar todas las columnas mediante un join por el índice de métricas
tabla_comparativa_ga = df_c1.join([df_c2, df_c3, df_c4, df_c5, df_c6], how='outer')

# Limpieza estética: Rellenar nulos de parámetros ausentes con un guión
tabla_comparativa_ga = tabla_comparativa_ga.fillna("-")

print("=======================================================================================")
print("             TABLA DE DESARROLLO EXPERIMENTAL: PARAMETRIZACIONES DEL GA")
print("=======================================================================================")
display(tabla_comparativa_ga)



# =====================================================================
# 3. EXPORTACIÓN DE LA TABLA COMPARATIVA A EXCEL
# =====================================================================

# Creamos una copia para no alterar la visualización en el Notebook
tabla_excel = tabla_comparativa_ga.copy()

# Opcional: Reemplazar los saltos de línea (\n) de los nombres de las columnas 
# por un espacio para que los encabezados queden limpios en Excel
tabla_excel.columns = [col.replace('\n', ' ') for col in tabla_excel.columns]

# Definir el nombre del archivo de salida
nombre_archivo_excel = "Desarrollo_Experimental_GA.xlsx"

# Exportar a Excel (incluyendo el índice, ya que ahí están los nombres de las métricas)
with pd.ExcelWriter(nombre_archivo_excel, engine='openpyxl') as writer:
    tabla_excel.to_excel(writer, sheet_name='Comparativa_GA')
    
    # Autoajuste automático del ancho de las columnas en el archivo de salida
    worksheet = writer.sheets['Comparativa_GA']
    for col in worksheet.columns:
        max_len = max(len(str(cell.value or '')) for cell in col)
        col_letter = col[0].column_letter
        worksheet.column_dimensions[col_letter].width = max(max_len + 3, 12)

print(f"\n[OK] Tabla exportada correctamente al archivo: '{nombre_archivo_excel}'")



NameError: name 'df_c2' is not defined

In [22]:
# Unificar o limpiar los valores NaN en la tabla comparativa
df_comparativo.loc[df_comparativo["Metrica"] == "Total alumnos", "Algoritmo Genético"] = 836.00

# O de forma general, cambiar los NaN restantes por un guion para que quede elegante
df_comparativo = df_comparativo.fillna("-")
df_experimentos.sort_values(by="mejor_coste").iloc[0]

# ============================================================
# CONSTRUCCIÓN DEL CUADRO COMPARATIVO UNIENDO DATASET
# ============================================================
df_res_sa.columns = ["Metrica", "Recocido Simulado"]
df_res_ga.columns = ["Metrica", "Algoritmo Genético"]

df_comparativo = pd.merge(df_res_sa, df_res_ga, on="Metrica", how="outer")
print("\n========================================================")
print("               TABLA COMPARATIVA DE MÉTODOS")
print("========================================================")
print(df_comparativo)



tabla_excel_c1 = df_comparativo.copy()

# Opcional: Reemplazar los saltos de línea (\n) de los nombres de las columnas 
# por un espacio para que los encabezados queden limpios en Excel
tabla_excel_c1.columns = [col.replace('\n', ' ') for col in tabla_excel_c1.columns]

# Definir el nombre del archivo de salida
nombre_archivo_excel = "Compara_Recocido_GA.xlsx"

# Exportar a Excel (incluyendo el índice, ya que ahí están los nombres de las métricas)
with pd.ExcelWriter(nombre_archivo_excel, engine='openpyxl') as writer:
    tabla_excel_c1.to_excel(writer, sheet_name='Comparativa_Recocido_GA')
    
    # Autoajuste automático del ancho de las columnas en el archivo de salida
    worksheet = writer.sheets['Comparativa_Recocido_GA']
    for col in worksheet.columns:
        max_len = max(len(str(cell.value or '')) for cell in col)
        col_letter = col[0].column_letter
        worksheet.column_dimensions[col_letter].width = max(max_len + 3, 12)

print(f"\n[OK] Tabla exportada correctamente al archivo: '{nombre_archivo_excel}'")

NameError: name 'df_comparativo' is not defined